In [ ]:
from Database.transform import *
from Database.embed_models import AsyncSapBERTEmbedder
from dotenv import dotenv_values
import sys

In [ ]:
env = dotenv_values("YOUR_PATH_TO_ENV_FILE")  # Update this to your actual .env file path
neo4j_dict = {
        "uri": env.get('NEO4J_HOST'),
        "username": env.get('NEO4J_USER'),
        "password": env.get('NEO4J_PASSWORD'),
        "snomed_release": env.get('SNOMED_RELEASE_PATH'),
}

In [ ]:
c_, s_, r_ = reformat_snomed(data_path=neo4j_dict["snomed_release"]) 

In [ ]:
async def main():
      if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
      await awrite_concept_nodes(
          lf=c_, 
          neo4j_url=neo4j_dict['uri'], 
          neo4j_user=neo4j_dict['username'], 
          neo4j_password=neo4j_dict['password'],
          batch_size = 10_000,
          )
      await awrite_relationships(r_, neo4j_dict['uri'], neo4j_dict['username'], neo4j_dict['password'])
      await awrite_description_nodes(
          lf= s_,
          embedding_model=embedding_model,
          neo4j_url=neo4j_dict['uri'],
          neo4j_user=neo4j_dict['username'],
          neo4j_password=neo4j_dict['password'],
          batch_size = 10_000,
      )
      

In [ ]:
delete_all_in_batches(uri=neo4j_dict['uri'], user=neo4j_dict['username'], password=neo4j_dict['password'])

In [ ]:
await main()